In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import yaml
from dataset import get_real_batch_groove

In [2]:
import torch
import sys
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from model import PQC

In [3]:
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
})

with open("./config.yaml", "r") as f:
    config = yaml.safe_load(f)

epochs = config["epochs"]
save_epoch = config["save_epoch"]
accuracy_file = config["accuracy_file"]
loss_dis_file = config["loss_dis_file"]
loss_gen_file = config["loss_gen_file"]
kl_div_file = config["kl_div_file"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
epoch_list = list(range(1, epochs+1, save_epoch))

Device: cuda


## 量子回路が生成するドラムパターンの可聴化

スネアとキックの生成結果を1つのMIDIファイルに統合し、MP3に変換します。

### 前提条件
- `mido` パッケージ（MIDIファイル生成）
- `fluidsynth` コマンド（MIDI→WAV変換）
- `ffmpeg` コマンド（WAV→MP3変換）
- GM SoundFont（`/usr/share/sounds/sf2/FluidR3_GM.sf2` など）

```bash
# 必要に応じてインストール
uv pip install mido
sudo apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg
```

In [4]:
# --- スネアとキックの両モデルをロードし、ドラムパターンを生成 ---
import os

# ============================================================
# ★ ここで生成する小節数を設定してください
#   1回のrunで16ステップ（1小節）分のパターンが生成されます。
#   num_measures 回 run を実行し、合計 16 × num_measures ステップの
#   ドラムパターンを作成します。
# ============================================================
num_measures = 8  # 生成する小節数（runの回数）

# 各楽器の実験ディレクトリ
snare_dir = './experiment/SnareN16_mps_L3_V2'
kick_dir = './experiment/KickN16_mps_L3_V2'

def load_model_and_generate(exp_dir, num_measures):
    """指定の実験ディレクトリからパラメータを読み込み、PQCでnum_measures回生成する
    
    1回のrunで16ステップ（=1小節）のパターンが1つ得られるので、
    num_measures回runして連結することで、num_measures小節分のパターンを構成する。
    """
    with open(f'{exp_dir}/param.yaml', 'r') as f:
        p = yaml.safe_load(f)

    length = p['length']
    size = length
    n_layers = p['n_layers']
    bs_dis = p['bs_dis']
    bs_gen = 1  # 1回のrunで1パターンずつ生成
    lr_gen = p['lr_gen']
    on_mps = p.get('on_mps', False)
    v_mps = p.get('v_mps', 2)

    if on_mps:
        model = PQC(n_layers, size, bs_dis, bs_gen, lr_gen, device, on_mps, v_mps)
    else:
        model = PQC(n_layers, size, bs_dis, bs_gen, lr_gen, device)

    theta_val = np.load(f'{exp_dir}/theta_val.npy')

    # num_measures回runして各小節のパターンを収集
    patterns = []
    for _ in range(num_measures):
        sample = model.run(params=theta_val[-1], mode='G')  # shape: (1, 16)
        patterns.append(sample[0])  # shape: (16,)

    return np.array(patterns)  # shape: (num_measures, 16)

# スネアとキックのパターンを生成
print(f'Generating {num_measures} measures ({num_measures} runs per instrument)...')
print(f'Total steps: 16 × {num_measures} = {16 * num_measures}')
print()

snare_samples = load_model_and_generate(snare_dir, num_measures)
kick_samples = load_model_and_generate(kick_dir, num_measures)

print(f'Snare patterns: {snare_samples.shape}')
print(f'Kick patterns: {kick_samples.shape}')
print()
for i in range(num_measures):
    print(f'--- Measure {i+1} ---')
    print(f'  Kick:  {kick_samples[i]}')
    print(f'  Snare: {snare_samples[i]}')

Generating 8 measures (8 runs per instrument)...
Total steps: 16 × 8 = 128

Snare patterns: (8, 16)
Kick patterns: (8, 16)

--- Measure 1 ---
  Kick:  [1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0]
  Snare: [0 1 0 0 0 1 0 1 1 0 1 1 0 0 1 0]
--- Measure 2 ---
  Kick:  [1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]
  Snare: [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0]
--- Measure 3 ---
  Kick:  [1 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0]
  Snare: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1]
--- Measure 4 ---
  Kick:  [1 0 0 1 0 0 1 0 0 0 0 1 0 1 0 1]
  Snare: [0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0]
--- Measure 5 ---
  Kick:  [1 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0]
  Snare: [0 0 1 1 1 1 0 0 1 0 0 1 1 0 1 1]
--- Measure 6 ---
  Kick:  [1 0 0 1 0 1 0 0 0 1 0 0 1 0 0 1]
  Snare: [0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0]
--- Measure 7 ---
  Kick:  [1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]
  Snare: [0 0 0 1 1 0 0 0 0 0 0 0 1 0 0 0]
--- Measure 8 ---
  Kick:  [1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]
  Snare: [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]


In [5]:
# --- スネアとキックを1つのMIDIファイルに統合 ---
import mido
from mido import MidiFile, MidiTrack, Message

# MIDI設定
BPM = 120
TICKS_PER_BEAT = 480  # 4分音符あたりのtick数
TICKS_PER_STEP = TICKS_PER_BEAT // 4  # 16分音符 = 1ステップ
VELOCITY = 100
NOTE_DURATION = TICKS_PER_STEP // 2  # ノートの長さ

# GM Drum Map: Kick=36, Snare=38 (Channel 10 = channel index 9)
KICK_NOTE = 36
SNARE_NOTE = 38
DRUM_CHANNEL = 9  # MIDIチャンネル10（0-indexed）

def create_combined_midi(kick_patterns, snare_patterns, output_path, bpm=BPM):
    """キックとスネアのパターンを1つのMIDIファイルに統合する"""
    mid = MidiFile(ticks_per_beat=TICKS_PER_BEAT)

    # テンポトラック
    tempo_track = MidiTrack()
    mid.tracks.append(tempo_track)
    tempo = mido.bpm2tempo(bpm)
    tempo_track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))
    tempo_track.append(mido.MetaMessage('track_name', name='Tempo', time=0))

    # ドラムトラック
    drum_track = MidiTrack()
    mid.tracks.append(drum_track)
    drum_track.append(mido.MetaMessage('track_name', name='Drums', time=0))

    # 全ステップのイベントを作成
    num_measures = len(kick_patterns)
    steps_per_measure = len(kick_patterns[0])
    total_steps = num_measures * steps_per_measure

    # フラットなパターンに変換
    kick_flat = kick_patterns.flatten()
    snare_flat = snare_patterns.flatten()

    # イベントリストを作成 (absolute_time, type, note, velocity)
    events = []
    for step in range(total_steps):
        abs_time = step * TICKS_PER_STEP
        if kick_flat[step] == 1:
            events.append((abs_time, 'note_on', KICK_NOTE, VELOCITY))
            events.append((abs_time + NOTE_DURATION, 'note_off', KICK_NOTE, 0))
        if snare_flat[step] == 1:
            events.append((abs_time, 'note_on', SNARE_NOTE, VELOCITY))
            events.append((abs_time + NOTE_DURATION, 'note_off', SNARE_NOTE, 0))

    # 絶対時間でソート
    events.sort(key=lambda x: (x[0], x[1] == 'note_off'))

    # デルタタイムに変換してトラックに追加
    current_time = 0
    for abs_time, msg_type, note, vel in events:
        delta = abs_time - current_time
        drum_track.append(Message(msg_type, channel=DRUM_CHANNEL, note=note,
                                  velocity=vel, time=delta))
        current_time = abs_time

    # End of track
    drum_track.append(mido.MetaMessage('end_of_track', time=TICKS_PER_STEP))
    tempo_track.append(mido.MetaMessage('end_of_track',
                                         time=total_steps * TICKS_PER_STEP + TICKS_PER_STEP))

    # 保存
    mid.save(output_path)
    print(f'MIDI file saved: {output_path}')
    return output_path

# 出力ディレクトリ
output_midi_dir = './audible_output'
os.makedirs(output_midi_dir, exist_ok=True)

midi_path = os.path.join(output_midi_dir, 'quantum_drums.mid')
create_combined_midi(kick_samples, snare_samples, midi_path, bpm=120)

print(f'\nGenerated {num_measures} measures of quantum drum pattern')
print(f'BPM: {BPM}, Steps per measure: 16 (16th notes)')

MIDI file saved: ./audible_output/quantum_drums.mid

Generated 8 measures of quantum drum pattern
BPM: 120, Steps per measure: 16 (16th notes)


In [6]:
# --- MIDI → MP3 変換 ---
import subprocess
import shutil

def midi_to_mp3(midi_path, mp3_path, soundfont=None):
    """fluidsynthとffmpegを使ってMIDIをMP3に変換する"""
    # fluidsynthとffmpegの存在確認
    if not shutil.which('fluidsynth'):
        print('ERROR: fluidsynth が見つかりません。')
        print('  sudo apt-get install -y fluidsynth fluid-soundfont-gm')
        return None
    if not shutil.which('ffmpeg'):
        print('ERROR: ffmpeg が見つかりません。')
        print('  sudo apt-get install -y ffmpeg')
        return None

    # SoundFont の自動検出
    if soundfont is None:
        sf_candidates = [
            '/usr/share/sounds/sf2/FluidR3_GM.sf2',
            '/usr/share/soundfonts/FluidR3_GM.sf2',
            '/usr/share/sounds/sf2/default-GM.sf2',
            '/usr/share/soundfonts/default.sf2',
        ]
        for sf in sf_candidates:
            if os.path.exists(sf):
                soundfont = sf
                break
    if soundfont is None:
        print('ERROR: SoundFont ファイルが見つかりません。')
        print('  sudo apt-get install -y fluid-soundfont-gm')
        return None

    print(f'Using SoundFont: {soundfont}')

    # MIDI → WAV (fluidsynth)
    wav_path = midi_path.replace('.mid', '.wav')
    cmd_synth = [
        'fluidsynth', '-ni', soundfont, midi_path,
        '-F', wav_path, '-r', '44100'
    ]
    print(f'Converting MIDI to WAV...')
    result = subprocess.run(cmd_synth, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'fluidsynth error: {result.stderr}')
        return None

    # WAV → MP3 (ffmpeg)
    cmd_mp3 = [
        'ffmpeg', '-y', '-i', wav_path,
        '-acodec', 'libmp3lame', '-ab', '192k', mp3_path
    ]
    print(f'Converting WAV to MP3...')
    result = subprocess.run(cmd_mp3, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'ffmpeg error: {result.stderr}')
        return None

    # WAV 中間ファイルを削除
    if os.path.exists(wav_path):
        os.remove(wav_path)

    print(f'MP3 file saved: {mp3_path}')
    return mp3_path

mp3_path = os.path.join(output_midi_dir, 'quantum_drums.mp3')
result = midi_to_mp3(midi_path, mp3_path)

if result:
    # Jupyter上で再生
    from IPython.display import Audio, display
    display(Audio(mp3_path))
else:
    print('\nMIDI → MP3 変換に失敗しました。')
    print('MIDIファイルは正常に生成されています:', midi_path)
    print('\n以下のコマンドで必要なパッケージをインストールしてください:')
    print('  sudo apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg')

Using SoundFont: /usr/share/sounds/sf2/FluidR3_GM.sf2
Converting MIDI to WAV...
Converting WAV to MP3...
MP3 file saved: ./audible_output/quantum_drums.mp3
